# Inference: Pretrained vs Finetuned KumoTabular

We run inference on the `_score` dataset, which was held out and never seen during finetuning. We compare the finetuned model vs the pretrained model on this set.

This notebook covers three inference approaches:

1. **Pretrained model** — baseline KumoTabular with no task-specific finetuning.
2. **Finetuned model (UC Model Registry)** *(recommended)* — loaded from Unity Catalog via `mlflow.pytorch.load_model()`.
3. **Finetuned model (Volume checkpoint)** — loaded from a `.pt` checkpoint stored in a UC Volume via `torch.load()`. Requires providing the parameter `finetune_run_id` to locate the checkpoint file. Included as a simpler alternative to get started before learning about MLflow.

All three use in-context learning (ICL): the model sees the training set as context, then predicts on the test set. For regression the model outputs 999 quantiles (q001–q999).

**Reproducibility:** KumoTabular's recipe pipeline applies stochastic transforms per estimator (random sign flips, column shuffles, etc.), so results vary across runs. We call `torch.manual_seed(SEED)` before each `.fit()` to fix the RNG state and ensure reproducible predictions.

**Metrics:** pinball loss at q100/q900, RMSE/MAE on the median (q500), and CRPS from the full quantile grid.

**Compute:** GPU (A10), environment version: AI v6.

## Configuration and libraries

In [0]:
%pip install git+https://github.com/NVIDIA/structured-data-models.git --quiet -q

In [0]:
dbutils.library.restartPython()

In [0]:
import importlib.util
import gc
import os
import sys
import time

import mlflow
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error
import torch
import sdm

In [0]:
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
COMMON_PATH = os.path.join(REPO_ROOT, "common")
CONFIG_PATH = os.path.join(COMMON_PATH, "config.py")

if not os.path.exists(CONFIG_PATH):
    raise FileNotFoundError(f"Could not find config.py at {CONFIG_PATH}")

spec = importlib.util.spec_from_file_location("project_config", CONFIG_PATH)
project_config = importlib.util.module_from_spec(spec)
spec.loader.exec_module(project_config)
CATALOG = project_config.CATALOG

SCHEMA = "default"
VOLUME_NAME = "finetuned_regression_models"
TABLE_NAME_PREFIX = "finetuning_regression"
TRAIN_TABLE = f"{CATALOG}.{SCHEMA}.{TABLE_NAME_PREFIX}_train"
SCORE_TABLE = f"{CATALOG}.{SCHEMA}.{TABLE_NAME_PREFIX}_score"
TARGET_COL = "flow_duration"
ID_COL = None  # RT-IoT2022 dataset has no ID column

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")
volume_path = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME_NAME}"

current_user = spark.sql("SELECT current_user()").collect()[0][0]
MLFLOW_EXPERIMENT_NAME = f"/Users/{current_user}/tabular-fm-databricks-finetune"
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

NUM_ESTIMATORS = 8
SEED = 42

print(f"Catalog/schema: {CATALOG}.{SCHEMA}")
print(f"Volume: {volume_path}")
print(f"MLflow experiment: {MLFLOW_EXPERIMENT_NAME}")

In [0]:
# Load train (for ICL context) and score (for evaluation)
train_pdf = spark.table(TRAIN_TABLE).toPandas().dropna(subset=[TARGET_COL])
score_pdf = spark.table(SCORE_TABLE).toPandas().dropna(subset=[TARGET_COL])

all_feature_cols = [c for c in train_pdf.columns if c not in (ID_COL, TARGET_COL)]
feature_cols = all_feature_cols

X_train = train_pdf[feature_cols].reset_index(drop=True)
y_train = train_pdf[[TARGET_COL]].reset_index(drop=True)
X_test = score_pdf[feature_cols].reset_index(drop=True)
y_test = score_pdf[TARGET_COL].values.astype(float)

# Cast numeric columns to float16 to match model precision
for df in (X_train, X_test):
    num_cols = df.select_dtypes(include=[np.number]).columns
    df[num_cols] = df[num_cols].astype(np.float16)
test_sample = score_pdf

device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Train (context): {len(X_train):,} rows")
print(f"Test:            {len(X_test):,} rows")
print(f"Features:        {len(feature_cols)}")
print(f"Device:          {device}")

## Metric helpers

In [0]:
QUANTILE_LEVELS = np.arange(1, 1000) / 1000.0  # 0.001, 0.002, ..., 0.999
Q100_IDX = 99   # q100 = 0.1 quantile
Q500_IDX = 499  # q500 = median
Q900_IDX = 899  # q900 = 0.9 quantile


def merge_quantile_estimators(preds_np: np.ndarray) -> np.ndarray:
    """Merge quantile predictions from multiple estimators by pooling and re-sorting.

    Each estimator outputs 999 quantiles (q001–q999). We pool all K×999 values
    per row, sort them, and re-pick 999 quantiles from the sorted pool via linear
    interpolation. This reconstructs the empirical CDF from all estimator samples
    before extracting quantiles.
    """
    if preds_np.ndim == 2:
        return preds_np
    if preds_np.shape[0] == 1:
        return preds_np.squeeze(0)
    n_est, n_rows, n_q = preds_np.shape
    pooled = preds_np.transpose(1, 0, 2).reshape(n_rows, n_est * n_q)
    pooled_sorted = np.sort(pooled, axis=1)
    positions = QUANTILE_LEVELS * (n_est * n_q - 1)
    lower = np.floor(positions).astype(int)
    upper = np.minimum(lower + 1, n_est * n_q - 1)
    frac = (positions - lower)[np.newaxis, :]  # (1, 999)
    return pooled_sorted[:, lower] * (1.0 - frac) + pooled_sorted[:, upper] * frac


def get_quantile_preds(pred_table) -> np.ndarray:
    """Extract (n_rows, 999) quantile numpy array from predict() output."""
    raw = pred_table.numerical.float().cpu().numpy()
    return merge_quantile_estimators(raw)


def compute_regression_metrics(preds: np.ndarray, y_true: np.ndarray) -> dict:
    """Compute regression metrics from (n_rows, 999) quantile predictions."""
    q100 = preds[:, Q100_IDX]
    q500 = preds[:, Q500_IDX]
    q900 = preds[:, Q900_IDX]

    rmse = float(np.sqrt(np.mean((q500 - y_true) ** 2)))
    mae = float(np.mean(np.abs(q500 - y_true)))

    def pinball(pred, actual, tau):
        error = actual - pred
        return float(np.mean(np.maximum(tau * error, (tau - 1) * error)))

    pinball_q100 = pinball(q100, y_true, 0.1)
    pinball_q900 = pinball(q900, y_true, 0.9)

    # CRPS from full quantile grid: mean pinball across all 999 quantile levels
    errors = y_true.reshape(-1, 1) - preds  # (n, 999)
    quantile_grid = QUANTILE_LEVELS.reshape(1, -1)  # (1, 999)
    pinball_grid = np.maximum(quantile_grid * errors, (quantile_grid - 1) * errors)
    crps = float(2.0 * pinball_grid.mean())

    return {
        "pinball_q100": pinball_q100,
        "pinball_q900": pinball_q900,
        "rmse_q500": rmse,
        "mae_q500": mae,
        "crps": crps,
    }

## Inference with Pretrained Model

In [0]:
regressor = sdm.models.KumoTabular(task="regression", device=device, size="small", pretrained=True)
regressor.half()  # Use float16 for faster inference and lower GPU memory

X_train_tensor = sdm.TableTensor.from_pandas(df=X_train, stypes=sdm.infer_stypes(X_train), device=device)
y_train_tensor = sdm.TableTensor.from_pandas(df=y_train, stypes=sdm.infer_stypes(y_train), device=device)

with mlflow.start_run(run_name="kumo_regression_pretrained"):
    mlflow.log_params({
        "vendor": "kumo",
        "model_type": "KumoTabularRegressor",
        "task": "rt_iot2022",
        "problem_type": "Regression",
        "finetuned": False,
        "train_table": TRAIN_TABLE,
        "score_table": SCORE_TABLE,
        "target_col": TARGET_COL,
        "n_features": len(feature_cols),
        "n_features_total": len(all_feature_cols),
        "n_train": len(X_train),
        "n_test": len(X_test),
        "device": device,
        "size": "small",
        "num_estimators": NUM_ESTIMATORS,
        "dtype": "float16",
    })

    t0 = time.time()
    torch.manual_seed(SEED)
    with torch.amp.autocast('cuda', dtype=torch.float16):
        regressor.fit(x=X_train_tensor, y=y_train_tensor, num_estimators=NUM_ESTIMATORS)
    fit_time = time.time() - t0

    del X_train_tensor, y_train_tensor
    torch.cuda.empty_cache()

    test_stypes = sdm.infer_stypes(X_test)
    t1 = time.time()
    X_test_tensor = sdm.TableTensor.from_pandas(df=X_test, stypes=test_stypes, device=device)
    with torch.amp.autocast('cuda', dtype=torch.float16):
        pred_table = regressor.predict(X_test_tensor)
    predict_time = time.time() - t1

    preds = get_quantile_preds(pred_table)
    metrics = compute_regression_metrics(preds, y_test)

    results_pdf = pd.DataFrame({
        TARGET_COL: y_test,
        "predicted_q500": preds[:, Q500_IDX],
        "predicted_q100": preds[:, Q100_IDX],
        "predicted_q900": preds[:, Q900_IDX],
    })

    mlflow.log_metrics({**metrics, "fit_time_s": fit_time, "predict_time_s": predict_time})
    mlflow.log_table(results_pdf.head(1000), "score_predictions.json")

    pretrained_run_id = mlflow.active_run().info.run_id

print(f"Run ID: {pretrained_run_id}")
print(f"Fit time:     {fit_time:.1f}s")
print(f"Predict time: {predict_time:.1f}s")
for k, v in metrics.items():
    print(f"{k}: {v:.6f}")
display(results_pdf.head(10))

In [0]:
# Free pretrained model before loading finetuned
_gpu_vars = [
    'regressor', 'ft_regressor', 'model',
    'X_train_tensor', 'y_train_tensor', 'X_test_tensor',
    'pred_table', 'preds',
]
for _v in _gpu_vars:
    if _v in globals():
        del globals()[_v]
for _name in list(globals()):
    _obj = globals()[_name]
    if isinstance(_obj, (torch.Tensor, torch.nn.Module)):
        del globals()[_name]
        continue
    try:
        if hasattr(_obj, 'device') and 'cuda' in str(_obj.device):
            del globals()[_name]
    except Exception:
        pass
gc.collect()
torch.cuda.empty_cache()
print(f"GPU cleaned: {torch.cuda.memory_allocated()/1e6:.0f} MiB allocated")

## Inference with Finetuned Model via UC Model Registry

In [0]:
# Load finetuned model from UC Model Registry (latest version)
from mlflow import MlflowClient

registered_model_name = f"{CATALOG}.{SCHEMA}.kumo_tabular_regression"
client = MlflowClient()
versions = client.search_model_versions(f"name='{registered_model_name}'")
latest_version = max(versions, key=lambda v: int(v.version)).version
model_uri = f"models:/{registered_model_name}/{latest_version}"

ft_regressor = mlflow.pytorch.load_model(model_uri)
ft_regressor = ft_regressor.to(device).half().eval()

print(f"Model: {registered_model_name}")
print(f"Version: {latest_version}")
print(f"URI:   {model_uri}")

In [0]:
X_train_tensor = sdm.TableTensor.from_pandas(df=X_train, stypes=sdm.infer_stypes(X_train), device=device)
y_train_tensor = sdm.TableTensor.from_pandas(df=y_train, stypes=sdm.infer_stypes(y_train), device=device)

with mlflow.start_run(run_name="kumo_regression_finetuned_mlflow"):
    mlflow.log_params({
        "vendor": "kumo",
        "model_type": "KumoTabularRegressor",
        "task": "rt_iot2022",
        "problem_type": "Regression",
        "finetuned": True,
        "model_source": "uc_model_registry",
        "registered_model_name": registered_model_name,
        "train_table": TRAIN_TABLE,
        "score_table": SCORE_TABLE,
        "target_col": TARGET_COL,
        "n_features": len(feature_cols),
        "n_features_total": len(all_feature_cols),
        "n_train": len(X_train),
        "n_test": len(X_test),
        "device": device,
        "num_estimators": NUM_ESTIMATORS,
        "dtype": "float16",
    })

    t0 = time.time()
    torch.manual_seed(SEED)
    with torch.amp.autocast('cuda', dtype=torch.float16):
        ft_regressor.fit(x=X_train_tensor, y=y_train_tensor, num_estimators=NUM_ESTIMATORS)
    fit_time_mlflow = time.time() - t0

    del X_train_tensor, y_train_tensor
    torch.cuda.empty_cache()

    test_stypes = sdm.infer_stypes(X_test)
    t1 = time.time()
    X_test_tensor = sdm.TableTensor.from_pandas(df=X_test, stypes=test_stypes, device=device)
    with torch.amp.autocast('cuda', dtype=torch.float16):
        pred_table_mlflow = ft_regressor.predict(X_test_tensor)
    predict_time_mlflow = time.time() - t1

    preds_mlflow = get_quantile_preds(pred_table_mlflow)
    metrics_mlflow = compute_regression_metrics(preds_mlflow, y_test)

    results_pdf_mlflow = pd.DataFrame({
        TARGET_COL: y_test,
        "predicted_q500": preds_mlflow[:, Q500_IDX],
        "predicted_q100": preds_mlflow[:, Q100_IDX],
        "predicted_q900": preds_mlflow[:, Q900_IDX],
    })

    mlflow.log_metrics({**metrics_mlflow, "fit_time_s": fit_time_mlflow, "predict_time_s": predict_time_mlflow})
    mlflow.log_table(results_pdf_mlflow.head(1000), "score_predictions.json")

    mlflow_finetuned_run_id = mlflow.active_run().info.run_id

print(f"Run ID: {mlflow_finetuned_run_id}")
print(f"Fit time:     {fit_time_mlflow:.1f}s")
print(f"Predict time: {predict_time_mlflow:.1f}s")
for k, v in metrics_mlflow.items():
    print(f"{k}: {v:.6f}")
display(results_pdf_mlflow.head(10))

In [0]:
# Free MLflow finetuned model before loading checkpoint model
_gpu_vars = [
    'ft_regressor', 'ft_regressor_ckpt',
    'X_train_tensor', 'y_train_tensor', 'X_test_tensor',
    'pred_table_mlflow', 'preds_mlflow', 'pred_table_ckpt', 'preds_ckpt',
]
for _v in _gpu_vars:
    if _v in globals():
        del globals()[_v]
for _name in list(globals()):
    _obj = globals()[_name]
    if isinstance(_obj, (torch.Tensor, torch.nn.Module)):
        del globals()[_name]
        continue
    try:
        if hasattr(_obj, 'device') and 'cuda' in str(_obj.device):
            del globals()[_name]
    except Exception:
        pass
gc.collect()
torch.cuda.empty_cache()
print(f"GPU cleaned: {torch.cuda.memory_allocated()/1e6:.0f} MiB allocated")

## Inference with Finetuned Model via Volume Checkpoint

We retrieve the run_id from the parameter finetune_run_id. Pick the run you want to infer with. 

In [0]:
# Read run_id from the finetune_run_id widget parameter
run_id = dbutils.widgets.get("finetune_run_id")
ft_run = mlflow.get_run(run_id)
FT_MODEL_SIZE = ft_run.data.params["model_size"]
ckpt_path = f"{volume_path}/kumo_tabular_{FT_MODEL_SIZE}_regression_finetuned_{run_id}.pt"

assert os.path.exists(ckpt_path), f"Checkpoint not found: {ckpt_path}"

print(f"Finetuning run: {run_id}")
print(f"Model size:     {FT_MODEL_SIZE}")
print(f"Checkpoint:     {ckpt_path}")
print(f"Best val CRPS:  {ft_run.data.metrics.get('ft_best_val_crps', 'N/A')}")

In [0]:
# Load finetuned model from volume checkpoint
ft_regressor_ckpt = sdm.models.KumoTabular(
    task="regression", device=device, size=FT_MODEL_SIZE, pretrained=False,
)
ft_regressor_ckpt.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=True))
ft_regressor_ckpt.half()
ft_regressor_ckpt.eval()

X_train_tensor = sdm.TableTensor.from_pandas(df=X_train, stypes=sdm.infer_stypes(X_train), device=device)
y_train_tensor = sdm.TableTensor.from_pandas(df=y_train, stypes=sdm.infer_stypes(y_train), device=device)

with mlflow.start_run(run_name="kumo_regression_finetuned_ckpt"):
    mlflow.log_params({
        "vendor": "kumo",
        "model_type": "KumoTabularRegressor",
        "task": "rt_iot2022",
        "problem_type": "Regression",
        "finetuned": True,
        "model_source": "volume_checkpoint",
        "finetune_checkpoint": ckpt_path,
        "finetune_run_id": run_id,
        "train_table": TRAIN_TABLE,
        "score_table": SCORE_TABLE,
        "target_col": TARGET_COL,
        "n_features": len(feature_cols),
        "n_features_total": len(all_feature_cols),
        "n_train": len(X_train),
        "n_test": len(X_test),
        "device": device,
        "size": FT_MODEL_SIZE,
        "num_estimators": NUM_ESTIMATORS,
        "dtype": "float16",
    })

    t0 = time.time()
    torch.manual_seed(SEED)
    with torch.amp.autocast('cuda', dtype=torch.float16):
        ft_regressor_ckpt.fit(x=X_train_tensor, y=y_train_tensor, num_estimators=NUM_ESTIMATORS)
    fit_time_ckpt = time.time() - t0

    del X_train_tensor, y_train_tensor
    torch.cuda.empty_cache()

    test_stypes = sdm.infer_stypes(X_test)
    t1 = time.time()
    X_test_tensor = sdm.TableTensor.from_pandas(df=X_test, stypes=test_stypes, device=device)
    with torch.amp.autocast('cuda', dtype=torch.float16):
        pred_table_ckpt = ft_regressor_ckpt.predict(X_test_tensor)
    predict_time_ckpt = time.time() - t1

    preds_ckpt = get_quantile_preds(pred_table_ckpt)
    metrics_ckpt = compute_regression_metrics(preds_ckpt, y_test)

    results_pdf_ckpt = pd.DataFrame({
        TARGET_COL: y_test,
        "predicted_q500": preds_ckpt[:, Q500_IDX],
        "predicted_q100": preds_ckpt[:, Q100_IDX],
        "predicted_q900": preds_ckpt[:, Q900_IDX],
    })

    mlflow.log_metrics({**metrics_ckpt, "fit_time_s": fit_time_ckpt, "predict_time_s": predict_time_ckpt})
    mlflow.log_table(results_pdf_ckpt.head(1000), "score_predictions.json")

    ckpt_finetuned_run_id = mlflow.active_run().info.run_id

print(f"Run ID: {ckpt_finetuned_run_id}")
print(f"Fit time:     {fit_time_ckpt:.1f}s")
print(f"Predict time: {predict_time_ckpt:.1f}s")
for k, v in metrics_ckpt.items():
    print(f"{k}: {v:.6f}")
display(results_pdf_ckpt.head(10))

## Why UC Registry and Checkpoint CRPS differ slightly

Both approaches load **identical weights** (same best checkpoint). Any residual metric gap comes from **stochastic ICL inference**: `.fit(num_estimators=8)` draws a different random context permutation per estimator. We set `torch.manual_seed(SEED)` before each `.fit()` call so that, given the same RNG state, the same context permutations are drawn and results are reproducible across re-runs of the same cell. The UC Registry and Checkpoint cells both seed identically, so their metrics should match closely.